In [ ]:
import sys

In [ ]:
!{sys.executable} -m pip install geopandas

In [ ]:
import geopandas as gpd
from pathlib import Path

In [ ]:
notebook_data_dir = Path("notebook_experimentation_data")
notebook_data_dir.mkdir(parents=True, exist_ok=True)

## Preprocesamiento
Generamos funcion para automatizar la seleccion de columnas utiles, asi como generar los GeoPackages para cada subcuenca. Si posteriormente cada sufijo requiere de un procesamiento especifico previo a la seleccion de columnas se pueden agregar aquí

In [ ]:
def cargar_capa(ruta_subcuenca, sufijo, columnas):
    archivo = list(ruta_subcuenca.glob(f"*{sufijo}.shp"))[0]

    df = gpd.read_file(archivo)

    if sufijo == "_to":
        df = df[df["NOM_MUNM"] == "Hermosillo"].copy()

    return df[columnas].copy()

In [ ]:
#Rutas subcuencas
subcuencas = {
    "San Miguel": notebook_data_dir / "Subc_San_Miguel" / "RH09" / "RH09D" / "RH09De",
    "Río Sonora-Hillo": notebook_data_dir / "Subc_R_Son_Hillo" / "RH09" / "RH09D" / "RH09Da",
    "La Poza": notebook_data_dir / "Subc_La_Poza" / "RH09" / "RH09D" / "RH09Di",
    "La Manga": notebook_data_dir / "Subc_La_Manga" / "RH09" / "RH09E" / "RH09Eb",
}

In [ ]:
columnas_utiles_hl = [
    "geometry",
    "ID",
    "CVE_SUBC",
    "CONDICION",
    "ORDER_1",
    "ID_DRENA"
]

columnas_utiles_dr = [
    "geometry",
    "ID",
    "CVE_SUBC",
    "CONDICION",
    "ID_DRENA",
    "ARBSUM"
]

columnas_utiles_subc = [
    "geometry",
    "ID",
    "CVE_SUBCUE"
]

columnas_utiles_ha = [
    "geometry",
    "IDBD",
    "FC",
    "CONDICION"
]

columnas_utiles_to = [
    "geometry",
    "FC",
    "CLASE",
    "TERMINO_GE",
    "NOMBRE"
]

In [ ]:
#Tenemos un diccionario de sufijos y sus correspondientes columnas utiles
columnas_utiles = {
    "_hl": columnas_utiles_hl,
    "_dr": columnas_utiles_dr,
    "_subc": columnas_utiles_subc,
    "_ha": columnas_utiles_ha,
    "_to": columnas_utiles_to
}

In [ ]:
#Crearemos un diccionario anidado en dataframes, contendra nombre_subcuenta -> sufijo -> datafram_limpio
dataframes = {}

for nombre_subcuenca, ruta_subcuenca in subcuencas.items():

    dataframes[nombre_subcuenca] = {}

    for sufijo, columnas in columnas_utiles.items():
        dataframes[nombre_subcuenca][sufijo] = cargar_capa(
            ruta_subcuenca,
            sufijo,
            columnas
        )

In [ ]:
ruta_salida =  Path("notebook_experimentation_data/interim")
ruta_salida.mkdir(exist_ok=True)

In [ ]:
#Crearemos un package por cada subcuenca
for nombre_subcuenca, capas in dataframes.items():

    nombre_gpkg = ruta_salida / f"{nombre_subcuenca}.gpkg"

    # Si el GeoPackage ya existe, lo eliminamos
    if nombre_gpkg.exists():
        nombre_gpkg.unlink()

    # Creamos nuevamente el GeoPackage con las capas limpias
    for sufijo, df in capas.items():

        nombre_capa = sufijo.lstrip("_")

        df.to_file(
            nombre_gpkg,
            layer=nombre_capa,
            driver="GPKG"
        )

## Lectura de los GeoPackages
Generamos funciones para la lectura automática de los GeoPackage de las subcuencas

In [ ]:
def leer_capa(ruta_gpkg, nombre_capa):
    return gpd.read_file(
        ruta_gpkg,
        layer=nombre_capa
    )

In [ ]:
def leer_geopackages(ruta, subcuencas):
#Funcion para leer todos los geopackage y almacenarlo en un diccionario anidado (dataframes)
    ruta = Path(ruta)
    dataframes = {}

    for nombre_subcuenca in subcuencas:

        nombre_gpkg = f"{nombre_subcuenca}.gpkg"
        ruta_gpkg = ruta / nombre_gpkg

        if not ruta_gpkg.exists():
            raise FileNotFoundError(
                f"No se encontró el GeoPackage de "
                f"'{nombre_subcuenca}': {ruta_gpkg}"
            )

        capas = {}

        for nombre_capa in gpd.list_layers(ruta_gpkg)["name"]:
            capas[nombre_capa] = leer_capa(
                ruta_gpkg,
                nombre_capa
        )

        dataframes[nombre_subcuenca] = capas

    return dataframes

In [ ]:
dataframes_recuperados = leer_geopackages("notebook_experimentation_data/interim",subcuencas)

In [ ]:
dataframes_recuperados.keys()

In [ ]:
for subcuenca, capas in dataframes_recuperados.items():
    print(subcuenca, "=", list(capas.keys()))

In [ ]:
dataframes_recuperados["San Miguel"]["hl"].head()

In [ ]:
dataframes_recuperados["Río Sonora-Hillo"]["hl"].head()

In [ ]:
dataframes_recuperados["La Poza"]["hl"].head()

In [ ]:
dataframes_recuperados["La Manga"]["hl"].head()